# Day 11/90 - Classical Machine Learning Models

Day 10 was about **how to evaluate a model**.

Day 11 is about:

> **What kind of model should I use, what does it actually do, and why might I choose one model instead of another?**

You do not need to memorize the mathematics of every algorithm.

For each model, focus on four questions:

```text
1. What problem does it solve?
2. How does it make a prediction?
3. What is it good at?
4. What limitation might make us choose something else?
```

This notebook is designed as a **visual explainer**.

The code used to create the graphs is hidden by default so you can focus on the concepts.

# 1. Start With the Problem

Before choosing a model, first ask:

> **What am I trying to predict?**

There are two basic problem types we need today.

## Regression

**Regression is used when the answer we want the model to predict is a number.**

Example:

```text
INPUTS
living area
bedrooms
bathrooms
year built

        ↓

REGRESSION MODEL

        ↓

OUTPUT
predicted price = $285,000
```

AI-engineering examples:

```text
predict request latency
predict API cost
predict delivery time
predict user spend
```

The output is a numerical value.

---

## Classification

**Classification is used when the answer we want is a category or class.**

Example:

```text
INPUT
"I was charged twice"

        ↓

CLASSIFICATION MODEL

        ↓

OUTPUT
billing_issue
```

Other examples:

```text
fraud / not fraud
spam / not spam
safe / unsafe
technical / billing / sales
```

So the first mental model is:

```text
Need a NUMBER?
-> Regression

Need a CATEGORY?
-> Classification
```

# 2. Baseline First

Before comparing machine-learning models, we need a **simple reference point**.

A baseline is a simple reference solution whose performance we measure first, so we can tell whether a new model actually improves the problem.

It answers:

> **Is the ML model actually learning something useful, or is it only slightly better than a very simple guess?**

## Example

Suppose the average sale price in the training data is:

```text
$180,000
```

A very simple regression baseline could ignore all house features and predict:

```text
House A -> $180,000
House B -> $180,000
House C -> $180,000
House D -> $180,000
```

This is obviously not a smart model.

That is exactly why it is useful.

It gives us a minimum performance level that a real ML model should beat.

Now suppose we evaluate it:

```text
Baseline MAE = $45,000
```

Then we train a real model.

### Case 1

```text
Baseline MAE = $45,000
ML model MAE = $44,000
```

The ML model is technically better, but only slightly.

That tells us:

```text
added complexity
↓
very small improvement
```

We may need better features, a different model, or a better understanding of the problem.

### Case 2

```text
Baseline MAE = $45,000
ML model MAE = $23,000
```

Now the ML model is much better.

That tells us:

```text
house features
↓
model learns useful patterns
↓
prediction error drops significantly
```

So the baseline gives meaning to the model's score.

Without a baseline:

```text
MAE = $23,000
```

does not tell us much by itself.

With a baseline:

```text
Baseline = $45,000 MAE
Model    = $23,000 MAE
```

we can clearly see that the model improved the result.

> **Baseline = the simple solution your model should meaningfully beat.**

The graph below uses an **illustrative example**. Lower MAE is better.

In [ ]:
#@title Baseline visualization { display-mode: "form" }
import plotly.io as pio
pio.renderers.default = 'notebook_connected'

import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Bar(
    x=["Simple baseline", "New ML model"],
    y=[45000, 23500],
    text=["$45,000", "$23,500"],
    textposition="outside",
    hovertemplate="%{x}<br>MAE: $%{y:,.0f}<extra></extra>"
))

fig.update_layout(
    title="Illustrative Example: Does the ML Model Beat the Baseline?",
    xaxis_title="Approach",
    yaxis_title="Mean Absolute Error ($)",
    height=460
)

fig.show()

### How to read this graph

The baseline is intentionally simple.

Its job is not to be the final solution.

Its job is to answer:

> **Did the machine-learning model actually improve anything?**

In this example:

```text
Baseline MAE = $45,000
ML model MAE = $23,500
```

Because **lower MAE is better**, the ML model makes much smaller prediction errors.

So we now have evidence that using the house features and training a model is worthwhile.

The same idea applies to AI systems.

For example:

```text
simple prompt success rate = 55%
agent workflow success rate = 78%
```

The agent clearly improves the result.

But:

```text
simple prompt = 76%
complex agent = 78%
```

may not justify all the extra complexity, latency, and cost.

That is the baseline-first mindset:

```text
simple solution
↓
measure it
↓
build a better solution
↓
compare
↓
keep the extra complexity only if it meaningfully helps
```

# 3. Linear Regression

## What is Linear Regression?

**Linear Regression is a model used to estimate a numerical value from one or more input features.**

It learns how changes in the inputs are associated with changes in the output.

Example:

```text
INPUTS
living area
bedrooms
bathrooms
garage spaces

        ↓

LINEAR REGRESSION

        ↓

OUTPUT
predicted house price
```

To understand the idea visually, start with only one input:

```text
living area -> sale price
```

If larger houses generally cost more, the model tries to learn a line that represents that overall relationship.

Use the graph below to explore:

```text
Data
-> the real examples

Line
-> the relationship learned by the model

Predictions
-> values the model predicts

Residuals
-> the model's prediction errors
```

In [ ]:
#@title Linear Regression visualization { display-mode: "form" }
import numpy as np
import plotly.graph_objects as go

rng = np.random.default_rng(11)

x = np.linspace(900, 3100, 34)
actual = 45000 + 105 * x + rng.normal(0, 28000, len(x))

# Fit the illustrative line with NumPy.
coef = np.polyfit(x, actual, 1)
pred = coef[0] * x + coef[1]
residual = actual - pred

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=x,
    y=actual,
    mode="markers",
    name="Actual houses",
    customdata=np.column_stack([pred, residual]),
    hovertemplate=(
        "Living area: %{x:,.0f} sq ft"
        "<br>Actual: $%{y:,.0f}"
        "<br>Predicted: $%{customdata[0]:,.0f}"
        "<br>Residual: $%{customdata[1]:,.0f}"
        "<extra></extra>"
    )
))

fig.add_trace(go.Scatter(
    x=x,
    y=pred,
    mode="lines",
    name="Learned line",
    hovertemplate="Living area: %{x:,.0f}<br>Prediction: $%{y:,.0f}<extra></extra>"
))

fig.add_trace(go.Scatter(
    x=x,
    y=pred,
    mode="markers",
    name="Predictions",
    hovertemplate="Prediction: $%{y:,.0f}<extra></extra>"
))

rx, ry = [], []
for idx in range(0, len(x), 2):
    rx.extend([x[idx], x[idx], None])
    ry.extend([pred[idx], actual[idx], None])

fig.add_trace(go.Scatter(
    x=rx,
    y=ry,
    mode="lines",
    name="Residuals",
    hoverinfo="skip"
))

buttons = [
    dict(label="Focus", method="update",
         args=[{"visible":[True,True,True,True]},
               {"title":"Linear Regression - Full View"}]),
    dict(label="Data", method="update",
         args=[{"visible":[True,False,False,False]},
               {"title":"Linear Regression - Raw Data"}]),
    dict(label="Line", method="update",
         args=[{"visible":[True,True,False,False]},
               {"title":"Linear Regression - Learned Relationship"}]),
    dict(label="Predictions", method="update",
         args=[{"visible":[True,True,True,False]},
               {"title":"Linear Regression - Predictions"}]),
    dict(label="Residuals", method="update",
         args=[{"visible":[True,True,False,True]},
               {"title":"Linear Regression - Prediction Errors"}]),
]

fig.update_layout(
    title="Linear Regression - Full View",
    xaxis_title="Living area (sq ft)",
    yaxis_title="Sale price ($)",
    updatemenus=[dict(type="buttons", direction="right", buttons=buttons, x=0, y=1.16)],
    height=560
)

fig.show()

### What is the graph actually showing?

Each dot represents a real example.

```text
x-axis -> living area
y-axis -> sale price
```

The line represents what the model has learned.

For example:

```text
living area = 2,000 sq ft
↓
look at the model's line
↓
predicted price
```

The model does **not** try to memorize every dot.

It tries to learn the overall relationship.

### What is a residual?

A residual is simply the prediction error:

```text
residual
=
actual value - predicted value
```

If:

```text
actual price    = $300,000
predicted price = $280,000
```

then:

```text
residual = $20,000
```

### Why use Linear Regression?

It is useful when you need to predict a number and want a model that is:

```text
simple
fast
easy to interpret
good as a starting baseline
```

### What is its limitation?

Linear Regression represents relationships using a linear combination of the features.

But real-world relationships can be more complicated.

For example:

```text
large house
AND
new construction
AND
large garage
```

may behave differently from what a simple linear relationship captures.

When we need more flexible rules and interactions, models such as Decision Trees become useful.

# 4. Logistic Regression

## What is Logistic Regression?

For now, think about **binary Logistic Regression**:

> **Logistic Regression helps solve problems where the goal is to estimate which of two possible outcomes is more likely based on the input features.**

Example:

```text
INPUTS
transaction amount
location
time
account activity

        ↓

LOGISTIC REGRESSION

        ↓

OUTPUT
probability of fraud = 0.82
```

The two possible outcomes could be:

```text
fraud / not fraud
spam / not spam
churn / stay
above $200k / below $200k
```

### It first predicts a probability

Suppose the model outputs:

```text
0.82
```

That means the model estimates a high probability of the positive class.

It does not yet mean:

```text
YES
```

We still need a **decision threshold**.

With:

```text
threshold = 0.50
```

we get:

```text
0.82 >= 0.50
↓
predict positive
```

So the flow is:

```text
INPUT FEATURES
↓
Logistic Regression
↓
PROBABILITY
↓
Decision threshold
↓
CLASS
```

Use the graph below to change the threshold and see how the final predictions change.

> Logistic Regression can also be extended to multiclass problems, but the two-outcome case is the easiest way to understand the core idea.

In [ ]:
#@title Logistic Regression visualization { display-mode: "form" }

import numpy as np
import plotly.graph_objects as go

scores = np.array([0.08, 0.16, 0.24, 0.32, 0.41, 0.48, 0.55, 0.63, 0.71, 0.79, 0.88, 0.95])
actual = np.array([0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1])
thresholds = [0.3, 0.5, 0.7]

fig = go.Figure()

for threshold in thresholds:
    predicted = (scores >= threshold).astype(int)
    correct = predicted == actual

    hover_text = []
    for probability, real_class, predicted_class, is_correct in zip(
        scores, actual, predicted, correct
    ):
        hover_text.append(
            f"<b>Probability:</b> {probability:.2f}<br>"
            f"<b>Actual class:</b> {'Positive' if real_class == 1 else 'Negative'}<br>"
            f"<b>Predicted class:</b> {'Positive' if predicted_class == 1 else 'Negative'}<br>"
            f"<b>Result:</b> {'Correct' if is_correct else 'Wrong'}"
        )

    fig.add_trace(
        go.Scatter(
            x=scores,
            y=actual,
            mode="markers",
            name=f"Threshold {threshold}",
            visible=(threshold == 0.5),
            marker=dict(
                size=15,
                symbol=["circle" if ok else "x" for ok in correct]
            ),
            text=hover_text,
            hovertemplate="%{text}<extra></extra>"
        )
    )

fig.add_shape(
    type="line",
    x0=0.5,
    x1=0.5,
    y0=-0.35,
    y1=1.35,
    line=dict(width=3, dash="dash")
)

buttons = []
for i, threshold in enumerate(thresholds):
    buttons.append(
        dict(
            label=f"Threshold {threshold}",
            method="update",
            args=[
                {"visible": [j == i for j in range(len(thresholds))]},
                {
                    "title": f"Logistic Regression: Threshold = {threshold}",
                    "shapes": [
                        dict(
                            type="line",
                            x0=threshold,
                            x1=threshold,
                            y0=-0.35,
                            y1=1.35,
                            line=dict(width=3, dash="dash")
                        )
                    ]
                }
            ]
        )
    )

fig.update_layout(
    title="Logistic Regression: Threshold = 0.5",
    xaxis=dict(
        title="Predicted probability of the positive class",
        range=[0, 1]
    ),
    yaxis=dict(
        title="Actual class",
        tickmode="array",
        tickvals=[0, 1],
        ticktext=["Actually Negative", "Actually Positive"],
        range=[-0.45, 1.45]
    ),
    updatemenus=[
        dict(
            type="buttons",
            direction="right",
            buttons=buttons,
            x=0,
            y=1.16
        )
    ],
    hovermode="closest",
    height=500,
    margin=dict(t=110)
)

fig.add_annotation(
    x=0.15,
    y=-0.30,
    text="Predicted NEGATIVE",
    showarrow=False
)

fig.add_annotation(
    x=0.85,
    y=-0.30,
    text="Predicted POSITIVE",
    showarrow=False
)

fig.show()

### How to read this graph

Each dot is one example.

```text
x-axis
-> probability produced by Logistic Regression

y-axis
-> what the example actually is
```

The dashed vertical line is the **decision threshold**.

With a threshold of `0.50`:

```text
probability < 0.50
-> predict Negative

probability >= 0.50
-> predict Positive
```

So the graph gives us four possibilities:

```text
Actually Positive + right of threshold
-> True Positive

Actually Positive + left of threshold
-> False Negative

Actually Negative + left of threshold
-> True Negative

Actually Negative + right of threshold
-> False Positive
```

In the graph:

```text
circle
-> correct prediction

X
-> wrong prediction
```

Hover over a point to see its exact probability, actual class, predicted class, and whether the prediction was correct.

Use the threshold buttons to see which examples switch classes.

The **probabilities do not change**. Only the cutoff used to turn probability into a final class changes.

# 5. Decision Trees

## What is a Decision Tree?

A Decision Tree is a model that repeatedly splits the data into smaller groups by asking questions about the input features, until it reaches a final prediction.

Think of it like a flowchart.

Example:

```text
Is living area > 2,000 sq ft?
            |
      ┌─────┴─────┐
     YES          NO
      |            |
Garage >= 2?   Built after 2000?
      |            |
   ┌──┴──┐      ┌──┴──┐
  YES   NO      YES    NO
```

Each question splits the data into smaller groups.

Eventually the model reaches a final prediction.

For example:

```text
living area > 2,000
↓ YES

garage spaces >= 2
↓ YES

predicted price
↓
$410,000
```

The important part:

> **We do not manually write these rules. The tree learns useful questions and thresholds from the training data.**

### Why use a Decision Tree?

A tree can naturally learn relationships such as:

```text
IF house is large
AND garage is large
AND house is newer
THEN price tends to be high
```

That makes trees good at handling:

```text
non-linear relationships
feature interactions
rule-like patterns
```

## What is the problem with one Decision Tree?

The same flexibility that makes Decision Trees useful can also make them unstable.

### Problem 1: Overfitting

A tree can keep creating more and more specific rules.

A shallow tree might learn:

```text
larger house -> higher price
```

A very deep tree may start learning extremely specific combinations from the training examples.

That can lead to:

```text
training data
-> excellent performance

new unseen data
-> worse performance
```

### Problem 2: High variance

A single tree can be sensitive to the exact training sample.

A small change in the training data can sometimes produce a noticeably different tree.

So the correct takeaway is **not**:

```text
Decision Trees are inaccurate.
```

It is:

> **A single Decision Tree is flexible, but it can overfit and can have high variance, making it less stable on unseen data.**

Use the graph below.

Watch what happens as the tree becomes deeper.

In [ ]:
#@title Decision Tree visualization { display-mode: "form" }
import numpy as np
import plotly.graph_objects as go

depth = np.arange(1, 21)

# Illustrative teaching curves.
train_mae = 52000 * np.exp(-0.20 * depth) + 5500
val_mae = 23000 + 23000*np.exp(-0.32*depth) + 430*np.maximum(depth-7,0)**1.5

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=depth, y=train_mae,
    mode="lines+markers",
    name="Training MAE",
    hovertemplate="Depth: %{x}<br>Training MAE: $%{y:,.0f}<extra></extra>"
))

fig.add_trace(go.Scatter(
    x=depth, y=val_mae,
    mode="lines+markers",
    name="Validation MAE",
    hovertemplate="Depth: %{x}<br>Validation MAE: $%{y:,.0f}<extra></extra>"
))

buttons = [
    dict(label="Both", method="update",
         args=[{"visible":[True,True]},
               {"title":"Decision Tree - Training vs Validation"}]),
    dict(label="Training", method="update",
         args=[{"visible":[True,False]},
               {"title":"Decision Tree - Training Error"}]),
    dict(label="Validation", method="update",
         args=[{"visible":[False,True]},
               {"title":"Decision Tree - Validation Error"}]),
]

fig.update_layout(
    title="Decision Tree - Training vs Validation",
    xaxis_title="Maximum tree depth",
    yaxis_title="MAE ($)",
    updatemenus=[dict(type="buttons", direction="right", buttons=buttons, x=0, y=1.16)],
    height=520
)

fig.show()

### How to read the graph

At first:

```text
increase tree depth
↓
model gets more flexible
↓
training error improves
↓
validation error may also improve
```

That is useful learning.

But eventually you may see:

```text
training error keeps falling
while
validation error stops improving or gets worse
```

Now the tree is becoming better at fitting the **training examples**, but not better at predicting **new examples**.

That is overfitting.

So we have a useful model:

```text
Decision Tree
-> flexible
-> learns complex rules
```

but also a weakness:

```text
one tree
-> can overfit
-> can be unstable
```

This leads naturally to **Random Forest**.

# 6. Random Forest

## What is a Random Forest?

**A Random Forest is a model that builds many different Decision Trees and combines their predictions.**

Instead of asking:

```text
What does one tree predict?
```

we ask:

```text
What do many trees predict?
```

For regression:

```text
Tree 1 -> $280k
Tree 2 -> $305k
Tree 3 -> $291k
Tree 4 -> $269k
Tree 5 -> $297k

        ↓

combine / average

        ↓

final prediction
```

For classification, the trees can vote on the class.

### Why do this?

Remember the problem with one Decision Tree:

```text
one tree
-> can be sensitive to training data
-> can have high variance
```

Random Forest creates variation between the trees and then combines them.

The mistakes made by individual trees can partly average out.

So:

```text
many varied trees
↓
combine predictions
↓
lower variance
↓
more stable prediction
```

Use the graph below to inspect the different tree predictions and their combined average.

In [ ]:
#@title Random Forest visualization { display-mode: "form" }
import plotly.graph_objects as go
import numpy as np

tree_predictions = np.array([280, 305, 291, 269, 297, 286, 312, 276, 294, 283]) * 1000
forest_prediction = tree_predictions.mean()

fig = go.Figure()

fig.add_trace(go.Bar(
    x=[f"Tree {i+1}" for i in range(len(tree_predictions))],
    y=tree_predictions,
    name="Individual tree prediction",
    hovertemplate="%{x}<br>Prediction: $%{y:,.0f}<extra></extra>"
))

fig.add_hline(
    y=forest_prediction,
    line_dash="dash",
    annotation_text=f"Forest average ≈ ${forest_prediction:,.0f}"
)

fig.update_layout(
    title="Illustrative Random Forest: Many Tree Predictions → One Average",
    xaxis_title="Individual tree",
    yaxis_title="Predicted value ($)",
    height=500
)

fig.show()

### What problem does Random Forest solve?

Random Forest mainly helps with the **instability of a single tree**.

The important progression is:

```text
Decision Tree
-> flexible
-> but can have high variance

Random Forest
-> keep tree flexibility
-> combine many varied trees
-> reduce variance
-> usually generalize more reliably
```

A subtle but important correction:

> Random Forest does not make a Decision Tree "more flexible." Decision Trees are already flexible.

What Random Forest mainly adds is **stability through averaging/voting**.

### Trade-off

Compared with one small tree, a Random Forest is:

```text
harder to interpret
more computationally expensive
```

So we use the extra complexity when the improvement is worthwhile.

# 7. Gradient Boosting

## What is Gradient Boosting?

**Gradient Boosting builds a strong model by adding smaller models one after another, where each new model focuses on improving the errors of the current model.**

Think:

```text
Model 1
makes predictions
↓
find what is still wrong
↓
Model 2
helps correct those errors
↓
find what is still wrong
↓
Model 3
improves again
↓
...
```

With tree-based boosting, those small models are usually shallow Decision Trees.

### Example intuition

Suppose:

```text
actual price     = $300,000
current prediction = $250,000
```

The current model is still off by:

```text
$50,000
```

The next learner is added to help reduce errors like that.

So the model improves **step by step**.

Use the graph below to see the illustrative error change as more boosting stages are added.

In [ ]:
#@title Gradient Boosting visualization { display-mode: "form" }
import numpy as np
import plotly.graph_objects as go

stages = np.arange(1, 101)
mae = 46000 - 23000*(1 - np.exp(-stages/18)) + 800*np.sin(stages/12)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=stages,
    y=mae,
    mode="lines",
    name="Validation MAE",
    hovertemplate="Boosting stage: %{x}<br>Validation MAE: $%{y:,.0f}<extra></extra>"
))

fig.update_layout(
    title="Illustrative Gradient Boosting: Sequential Improvement",
    xaxis_title="Number of boosting stages",
    yaxis_title="Validation MAE ($)",
    height=480
)

fig.show()

### Random Forest vs Gradient Boosting

Both use multiple trees, but the reason is different.

### Random Forest

```text
Tree 1 ─┐
Tree 2 ─┤
Tree 3 ─┼──→ combine predictions
Tree 4 ─┤
Tree 5 ─┘
```

Think:

> **Build many varied trees and average/vote to reduce instability.**

### Gradient Boosting

```text
Tree 1
↓
find remaining errors
↓
Tree 2 improves them
↓
find remaining errors
↓
Tree 3 improves again
```

Think:

> **Build learners sequentially so each new one improves the current model.**

Popular systems based on boosting ideas include:

```text
XGBoost
LightGBM
CatBoost
```

### Why use Gradient Boosting?

It can be extremely strong on structured/tabular data.

### What is the trade-off?

Compared with a simple model, boosting can be:

```text
more sensitive to hyperparameters
harder to interpret
more computationally expensive
capable of overfitting if pushed too far
```

For Day 11, the **sequential improvement idea** is the important part.

# 8. K-Nearest Neighbors

## What is KNN?

**K-Nearest Neighbors predicts a new example by looking at the most similar examples it already knows.**

For classification:

```text
NEW EXAMPLE
↓
find the K closest training examples
↓
look at their classes
↓
majority vote
↓
prediction
```

Suppose:

```text
K = 5
```

and the five nearest examples are:

```text
Class A
Class A
Class B
Class A
Class B
```

Then:

```text
Class A = 3 votes
Class B = 2 votes
```

Prediction:

```text
Class A
```

### What does K mean?

`K` simply means:

> **How many nearby examples should participate in the decision?**

Use the buttons on the graph to change:

```text
K = 1
K = 3
K = 5
K = 9
```

and watch which examples become the selected neighbors.

In [ ]:
#@title KNN visualization { display-mode: "form" }
import numpy as np
import plotly.graph_objects as go

rng = np.random.default_rng(8)

a = rng.normal([3.2, 6.4], [0.9, 0.75], size=(18,2))
b = rng.normal([7.0, 3.2], [0.9, 0.8], size=(18,2))

points = np.vstack([a,b])
labels = np.array(["Class A"]*len(a) + ["Class B"]*len(b))
query = np.array([5.0,4.8])

d = np.sqrt(((points-query)**2).sum(axis=1))

fig = go.Figure()

for class_name in ["Class A","Class B"]:
    mask = labels == class_name
    fig.add_trace(go.Scatter(
        x=points[mask,0], y=points[mask,1],
        mode="markers",
        name=class_name,
        hovertemplate=f"{class_name}<br>Feature 1: %{{x:.2f}}<br>Feature 2: %{{y:.2f}}<extra></extra>"
    ))

fig.add_trace(go.Scatter(
    x=[query[0]], y=[query[1]],
    mode="markers+text",
    text=["NEW"],
    textposition="top center",
    name="New example",
    marker=dict(size=18, symbol="star"),
    hovertemplate="New example<br>Feature 1: %{x:.2f}<br>Feature 2: %{y:.2f}<extra></extra>"
))

ks = [1,3,5,9]
predictions = {}

for k in ks:
    idx = np.argsort(d)[:k]
    vals, counts = np.unique(labels[idx], return_counts=True)
    prediction = vals[np.argmax(counts)]
    predictions[k] = prediction

    fig.add_trace(go.Scatter(
        x=points[idx,0],
        y=points[idx,1],
        mode="markers",
        name=f"{k} nearest",
        marker=dict(size=19, symbol="circle-open", line=dict(width=3)),
        visible=(k==5),
        customdata=np.column_stack([labels[idx], d[idx]]),
        hovertemplate=(
            "Neighbor"
            "<br>Class: %{customdata[0]}"
            "<br>Distance: %{customdata[1]:.2f}"
            "<extra></extra>"
        )
    ))

# traces 0,1 = classes; 2=query; 3..6 = K highlights
buttons = []
for pos,k in enumerate(ks):
    vis = [True,True,True] + [j==pos for j in range(4)]
    buttons.append(dict(
        label=f"K = {k}",
        method="update",
        args=[
            {"visible":vis},
            {"title":f"KNN - K={k} - Prediction: {predictions[k]}"}
        ]
    ))

fig.update_layout(
    title=f"KNN - K=5 - Prediction: {predictions[5]}",
    xaxis_title="Feature 1",
    yaxis_title="Feature 2",
    updatemenus=[dict(type="buttons", direction="right", buttons=buttons, x=0, y=1.16)],
    height=540
)

fig.show()

### What should you understand?

With a small `K`:

```text
very few neighbors decide
↓
very local prediction
↓
can be sensitive to noise
```

With a larger `K`:

```text
more neighbors participate
↓
decision becomes smoother
↓
very local patterns may be lost
```

There is no universally correct K.

It is a hyperparameter we can evaluate.

---

## Why does feature scaling matter so much for KNN?

KNN decides which examples are "near" by calculating **distance**.

Suppose:

```text
living area = 1,800
bedrooms    = 3
```

The raw numerical scales are very different.

Without scaling, the feature containing values in the thousands may dominate the distance calculation simply because of its numerical size.

So:

```text
KNN
↓
depends on distance
↓
distance depends on numerical scale
↓
feature scaling matters
```

This is one of the clearest practical examples of why we learned feature scaling on Day 10.

The similarity idea will also appear again later with **embeddings and vector search**.

# 9. Feature Engineering

## What is Feature Engineering?

**Feature engineering means creating a more useful input for the model from the data we already have.**

Suppose our dataset contains:

```text
year_built = 1995
year_sold  = 2010
```

We can create:

```text
house_age_at_sale = 15
```

We have not created new real-world information.

We have reorganized existing information into a form that may be easier for the model to use.

Another example:

```text
living area = 2,000 sq ft
bedrooms    = 4
```

Create:

```text
area_per_bedroom = 500 sq ft
```

So:

```text
RAW FEATURES
↓
use knowledge about the problem
↓
CREATE / TRANSFORM FEATURES
↓
potentially more useful model inputs
```

### Why do this?

Sometimes the information the model needs is present in the raw data but not expressed clearly.

A useful engineered feature can expose that relationship directly.

> **Better features can sometimes help more than choosing a more complicated model.**

Feature engineering does **not** mean creating random columns until the score improves.

The new feature should have a reasonable connection to the problem.

# 10. Parameters vs Hyperparameters

These terms sound similar, but the difference is simple.

## Parameters

**Parameters are values the model learns from the training data.**

You provide:

```text
training examples
↓
model learns
↓
parameters
```

For example, Linear Regression learns the coefficients that connect features to the prediction.

We do not manually choose those learned coefficients.

---

## Hyperparameters

**Hyperparameters are settings we choose that control how the model learns or how complex it is.**

Examples:

```text
Decision Tree
-> max_depth

Random Forest
-> number of trees

KNN
-> number of neighbors K

Gradient Boosting
-> learning rate
-> number of boosting stages
```

Mental model:

```text
YOU
↓
choose hyperparameters
↓
training begins
↓
MODEL
↓
learns parameters from data
```

The easiest sentence to remember is:

> **Parameters are learned by the model. Hyperparameters are configured by us.**

# 11. Hyperparameter Tuning

## What is Hyperparameter Tuning?

**Hyperparameter tuning means trying different model settings and using validation performance to decide which setting works better.**

Suppose a Decision Tree has:

```text
max_depth
```

This controls how deep the tree is allowed to grow.

We might try:

```text
depth = 2
depth = 4
depth = 6
depth = 10
```

For each setting:

```text
choose hyperparameter
↓
train model
↓
evaluate on validation data
↓
record result
```

Then compare the results.

The graph below shows this idea visually.

In [ ]:
#@title Hyperparameter tuning visualization { display-mode: "form" }
import numpy as np
import plotly.graph_objects as go

depths = np.array([1,2,3,4,5,6,8,10,12,15])
mae = np.array([47000,36500,30500,26900,24700,23900,24400,25700,27300,30200])

best = np.argmin(mae)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=depths,
    y=mae,
    mode="lines+markers",
    name="Validation MAE",
    hovertemplate="max_depth: %{x}<br>Validation MAE: $%{y:,.0f}<extra></extra>"
))

fig.add_trace(go.Scatter(
    x=[depths[best]],
    y=[mae[best]],
    mode="markers+text",
    text=["Best validation result"],
    textposition="top center",
    name="Selected depth",
    marker=dict(size=16),
    hovertemplate="Selected depth: %{x}<br>MAE: $%{y:,.0f}<extra></extra>"
))

fig.update_layout(
    title="Illustrative Hyperparameter Tuning",
    xaxis_title="Decision Tree max_depth",
    yaxis_title="Validation MAE ($)",
    height=480
)

fig.show()

### How do we interpret the graph?

A tree that is too shallow may be too simple:

```text
too little complexity
↓
underfitting
```

Making it deeper can improve performance.

But if we keep increasing complexity:

```text
tree gets extremely specific
↓
possible overfitting
↓
validation performance can get worse
```

So tuning is not:

```text
choose the largest value
```

It is:

> **Find a reasonable setting that performs well on validation data.**

And the Day 10 rule still applies:

```text
TRAIN
-> learn model

VALIDATION
-> choose settings

TEST
-> final evaluation only
```

Do not tune on the final test set.

# 12. Why Do We Move From One Model to Another?

This is the most important part of Day 11.

Do not memorize models as an arbitrary list.

Understand the **problem → limitation → next idea** relationship.

## Linear Regression → Decision Tree

```text
Linear Regression
-> simple and interpretable

Problem:
relationship may be non-linear
or depend on feature interactions

↓

Decision Tree
-> learns flexible if/else-style rules
```

## Decision Tree → Random Forest

```text
Decision Tree
-> flexible

Problem:
one tree can overfit
and have high variance

↓

Random Forest
-> combine many varied trees
-> reduce variance
-> improve stability
```

## Random Forest → Gradient Boosting

```text
Random Forest
-> many trees built mostly independently
-> combine their answers

Another approach:

Gradient Boosting
-> add learners sequentially
-> each new learner improves remaining errors
```

This does **not** mean Gradient Boosting is automatically better.

It is a different strategy.

## Raw Features → Feature Engineering

```text
Problem:
useful information may be hidden in raw columns

↓

Feature Engineering
-> create a clearer input representation
```

## Any Model → Hyperparameter Tuning

```text
Problem:
one model configuration may not suit the data

↓

Tuning
-> compare reasonable settings
-> use validation results
```

The engineering mindset is:

> **Identify what the current approach is missing, then add complexity for a reason.**

# 13. Why This Matters for AI Engineering

You may be planning to work with:

```text
LLMs
RAG
agents
multimodal systems
```

So why learn these classical models?

Because a production AI system can contain **many smaller prediction problems**.

It may look like:

```text
USER REQUEST
↓
intent classifier
↓
router
↓
retrieval
↓
ranking
↓
LLM
↓
quality / risk check
↓
RESPONSE
```

Not every step needs an LLM.

## Example: routing

```text
"I was charged twice"
↓
classification model
↓
billing_issue
↓
billing agent
```

## Example: latency prediction

```text
prompt length
context size
tool count
model type
↓
regression model
↓
predicted latency = 4.2 sec
```

## Example: risk prediction

```text
request features
↓
classification model
↓
high risk / low risk
```

A classical model can sometimes be:

```text
faster
cheaper
more deterministic
easier to debug
easier to evaluate
```

The goal is not:

```text
use classical ML everywhere
```

or:

```text
use an LLM everywhere
```

It is:

> **Choose the simplest approach that solves the actual problem reliably.**

# 14. What an AI Engineer Actually Needs to Know

| Concept | Plain-English meaning |
|---|---|
| **Baseline** | A simple reference solution your model should meaningfully beat |
| **Linear Regression** | Estimate a numerical value from input features |
| **Logistic Regression** | Estimate which of two outcomes is more likely from input features |
| **Decision Tree** | Make a prediction by asking a learned sequence of questions |
| **Decision Tree weakness** | One tree can overfit and be sensitive to the training sample |
| **Random Forest** | Combine many varied trees to make predictions more stable |
| **Gradient Boosting** | Add models sequentially so each new one improves remaining errors |
| **KNN** | Predict using the most similar nearby examples |
| **Feature Engineering** | Turn existing data into inputs that express useful information more clearly |
| **Parameter** | A value learned by the model |
| **Hyperparameter** | A model setting chosen by us |
| **Hyperparameter Tuning** | Compare settings using validation data |

For each topic, you should be able to answer:

```text
What does it do?
Why would I use it?
What is its weakness?
What might I try next?
```

You do not need to derive every algorithm mathematically yet.

# 15. Day 11 Mental Model

Start here:

```text
What am I predicting?
        ↓
NUMBER or CATEGORY?
        ↓
build a simple BASELINE
        ↓
try a simple model
        ↓
evaluate
        ↓
is it good enough?
     ↙       ↘
   YES        NO
    ↓          ↓
  stop    what is the limitation?
               ↓
      choose an approach that
      addresses that limitation
               ↓
            evaluate again
               ↓
          tune if necessary
               ↓
          final test
```

The model relationships:

```text
Linear Regression
simple numerical prediction
        ↓
need flexible non-linear rules?
        ↓
Decision Tree
        ↓
tree overfits / high variance?
        ↓
Random Forest
combine many trees for stability
```

A different tree strategy:

```text
Gradient Boosting
-> sequentially improve remaining errors
```

A similarity-based strategy:

```text
KNN
-> use nearby examples
```

And across all models:

```text
good features
baseline
proper validation
hyperparameter tuning only when needed
```


**Happy Learning :)**